# INFO 2950 Final Project: Data Cleaning Notebook (Phase 3)

**Goal.** Turn two raw chart sources (weekly Billboard Hot 100 charts and weekly Spotify US Top 200 charts) into one clean table with one row per song, holding the early signals and outcomes our analysis needs.

This notebook is the **Data collection and cleaning section of our Phase 2 notebook**, moved into its own file as the Phase 3 rubric asks. Our Phase 3 analysis notebook imports the CSV files this notebook saves and does no cleaning of its own.

**How to read the changes.** The Phase 2 code and text below are unchanged, except:
- Code lines we changed or added are marked with a `# Phase 3:` comment.
- Text we added starts with **Phase 3 change:**.
- The section **New in Phase 3** at the end holds the new cleaning steps our preregistered hypotheses need.

**Summary of Phase 3 changes**
1. Cleaned files are saved to a `cleaned/` folder that we upload to `Data/cleaned/` in our repo, instead of Colab's temporary `/content/` folder, which is wiped when the session ends.
2. The matching key now handles accents ("Beyoncé"), curly apostrophes, and Billboard's "Feauring" typo. These caused 3 of the 4 missed top-40 debuts in Phase 2 (one of our Phase 2 reviewer questions).
3. We load 23 Spotify weeks (through Oct 8, 2026) instead of 20, and the Spotify settings now accept several date ranges so more weeks can be added with one line.
4. New columns for Hypothesis 3 (week-2 Spotify streams and the front-loading ratio F), Hypothesis 2 (the `established` flag), and our linear regressions (`log_weeks`).
5. We also save the cleaned weekly Billboard and Spotify tables, which the analysis notebook uses for week-by-week plots.
6. We fixed three cleaning bugs we found while checking our results. Songs are now grouped by a `song_id` (the title and first artist, simplified), because Billboard sometimes rewrites a credit partway through a run, which had split 60 songs in two and cut the first part's weeks short. Week 2 is now the chart exactly one week after the debut; before, it was the song's 2nd week on the chart, which for about 130 songs was a re-entry weeks later. In the last 3 charts the scraped weeks-on-chart count resets for a few songs, so a song's total can no longer be lower than the weeks we see it.
7. Two new columns respond to TA feedback: `spotify_lag_days`, which moves the 9-day match check here from the analysis notebook, and two stricter versions of `established`.

# Data collection and cleaning

## How we collected the data

We used two chart sources for songs on the US charts.

**Billboard Hot 100 (weekly).** We used the open archive `hot-100-current.csv` from the `utdata/rwd-billboard-data` GitHub repo. It is run by Christian McDonald, a journalism professor at UT Austin, for his data journalism courses. An automated script on GitHub copies (scrapes) the chart from billboard.com every week and saves it. Each row is one song on one weekly chart, with its rank, last week's rank, peak position so far, and weeks on chart so far. We downloaded the file on Oct 14, 2026, and the newest chart in it is from 2026-10-17.

The repo's README says that charts from 2022 onward are scraped straight from billboard.com, but the older years were pieced together from other datasets (including Kaggle and data.world) plus some manual gap filling. We only wanted data we could trace back to Billboard, so **we only use charts from 2022-01-01 on**.

**Spotify US Top 200 (weekly).** We downloaded the weekly US Top 200 charts as CSV files from Spotify's official charts site, charts.spotify.com, using a Spotify account. We downloaded 20 weeks of charts, from the chart dated May 7, 2026 to the chart dated September 17, 2026, one file per week, and saved them in our team's public GitHub repo: https://github.com/harrylin1015/info2950-chart-longevity/tree/main/Data/spotify_weekly_us

**Why we switched sources.** Our first plan was to scrape billboard.com with `billboard.py` and Spotify with `fycharts`, but both stopped working. `fycharts` was made for the old spotifycharts.com site, and charts.spotify.com now needs a login. We also decided not to use Kaggle datasets because we could not check where their data came from.

**Limits of how we collected it.** We downloaded the Spotify files by hand, so we have far fewer Spotify weeks than Billboard weeks, and anything that uses Spotify data only covers those weeks. Weekly charts give us streams from a track's first week on the Top 200, not its first three days.

**Phase 3 change:** we have since downloaded 3 more Spotify weeks, so we now use 23 weekly files, from the chart dated May 7, 2026 to the chart dated October 8, 2026.

In [1]:
# Data lives in our team's public GitHub repo, so the notebook runs for anyone with no Drive setup
import pandas as pd
import numpy as np  # Phase 3: needed in the Billboard cell below

# Folder with hot-100-current.csv and the spotify_weekly_us folder
DATA_FOLDER = "https://raw.githubusercontent.com/harrylin1015/info2950-chart-longevity/main/Data/"
# Phase 3: save to a "cleaned" folder, then upload it to Data/cleaned/ in our repo for the analysis notebook
OUTPUT_FOLDER = "cleaned/"
import os  # Phase 3
os.makedirs(OUTPUT_FOLDER, exist_ok=True)  # Phase 3

# Only keep Billboard charts from 2022 on, since those are scraped straight from billboard.com
STUDY_START = pd.Timestamp("2022-01-01")

## Cleaning: Billboard Hot 100

Steps:
1. Load the file and rename the columns so they are easier to read.
2. Turn `week` into a date and `NA` into missing values.
3. Drop `wks_at_no1`. It is missing for almost every row, and no analysis uses it.
4. Remove rows where the same song appears twice in the same chart (same week, title, and artist). All 13 are from October 1990, so none fall in our window. We also checked that no two songs share a rank in any chart from 2022 on.
5. Keep only 2022 on.
6. Phase 3: replace the 0 the archive uses for "not on last week's chart" with a missing value.
7. Phase 3: give each song a `song_id` (title and first artist, lowercased, without accents or punctuation), so a credit that changes partway through a run, like "Jessie Murph" becoming "jessie murph" on Aug 2, 2025, no longer splits one song in two.

**Phase 3 change:** step 5 still applies to every analysis, with one exception. The `established` flag for Hypothesis 2 looks back 3 years, so it reads pre-2022 charts from the raw file (`bb_raw`). See "New in Phase 3" below.

In [2]:
# Load the Billboard file, rename columns, drop duplicates, and keep charts from 2022 on
bb_raw = pd.read_csv(DATA_FOLDER + "hot-100-current.csv", na_values=["NA"])
print("raw rows:", len(bb_raw))
print("columns:", list(bb_raw.columns))

bb = bb_raw.rename(columns={
    "chart_week": "week",
    "current_week": "rank",
    "performer": "artist",
    "last_week": "last_rank",
    "peak_pos": "peak_pos_to_date",
    "wks_on_chart": "weeks_on_chart_to_date",
}).drop(columns="wks_at_no1")

bb["week"] = pd.to_datetime(bb["week"])

rows_before = len(bb)
bb = bb.drop_duplicates(["week", "title", "artist"], keep="first")
print("duplicate rows removed:", rows_before - len(bb))

bb = bb[bb["week"] >= STUDY_START].sort_values(["week", "rank"]).reset_index(drop=True)
# Phase 3: from 2022 on, the archive writes 0 (not NA) when a song was not on last week's chart
bb["last_rank"] = bb["last_rank"].replace(0, np.nan)

# Phase 3: one ID per song. Billboard sometimes rewrites a song's credit partway through its run
# ("Charli XCX" -> "Charli xcx", "Karol G & Drake" -> "Karol G With Drake", "HUNTR/X" -> "HUNTR/X: EJAE"),
# which split one run into two "songs" and cut the first one's weeks short.
# The ID is the title plus the first-listed artist, lowercased, without accents, spaces, or punctuation.
def simplify(s):
    return (s.str.lower().str.normalize("NFKD").str.encode("ascii", errors="ignore").str.decode("ascii")
             .str.replace(r"[^a-z0-9]", "", regex=True))

lead = (bb["artist"].str.lower().str.replace(" feauring ", " featuring ", regex=False)
          .str.split(r" featuring | with | & |, | x |: ", regex=True).str[0])
bb["song_id"] = simplify(bb["title"]) + "|" + simplify(lead)
print("songs split by a credit change, now merged:", bb.groupby(["title", "artist"]).ngroups - bb["song_id"].nunique())

print("rows from 2022 on:", len(bb), "| weeks:", bb["week"].nunique())
print("date range:", bb["week"].min().date(), "to", bb["week"].max().date())
print("weeks without exactly 100 rows:", (bb.groupby("week").size() != 100).sum())

raw rows: 355900
columns: ['chart_week', 'current_week', 'title', 'performer', 'last_week', 'peak_pos', 'wks_on_chart', 'wks_at_no1']
duplicate rows removed: 13


songs split by a credit change, now merged: 60
rows from 2022 on: 25100 | weeks: 251
date range: 2022-01-01 to 2026-10-17
weeks without exactly 100 rows: 0


### One row per song

The Hot 100 file has one row per song per week, but our question is about whole songs, so we collapse it to one row per song (one `song_id`). Two things matter for measuring longevity.

- **Songs that started before our data.** If a song was already on the chart before 2022, the first week we see is not its real debut. We only call a song a `new_entry` if its weeks-on-chart count was 1 the first time it shows up.
- **Songs that are still charting.** If a song is on the chart in the latest week, its run is not over yet, so its total weeks is not final. We mark these `still_charting`. Songs that are both a new entry and finished are marked `complete_run`.

`weeks_on_chart_total` comes from Billboard's own running count, which includes re-entries, so it can be bigger than the number of weeks in a row. We also keep `weeks_in_window`, which is how many weeks we actually see. **Phase 3 change:** in the last three charts (Oct 3 to 17, 2026), the scraped count resets to 1 for a few songs, so we never let `weeks_on_chart_total` fall below `weeks_in_window`.

In [3]:
# Collapse the weekly table to one row per song and flag incomplete runs
latest_week = bb["week"].max()

songs = (
    bb.sort_values("week")
      .groupby("song_id", as_index=False)  # Phase 3: group by song ID, not the exact title and artist text
      .agg(
          title=("title", "first"),    # Phase 3
          artist=("artist", "first"),  # Phase 3
          first_week=("week", "min"),
          last_week=("week", "max"),
          weeks_in_window=("week", "nunique"),
          weeks_on_chart_total=("weeks_on_chart_to_date", "max"),
          count_at_first_obs=("weeks_on_chart_to_date", "first"),
          debut_rank=("rank", "first"),
          best_rank=("rank", "min"),
          peak_pos=("peak_pos_to_date", "min"),
      )
)

# Phase 3: in the last 3 charts (Oct 3-17, 2026) the scraped weeks-on-chart count resets for a few songs,
# so we never let a song's total fall below the number of weeks we actually see it
songs["weeks_on_chart_total"] = songs[["weeks_on_chart_total", "weeks_in_window"]].max(axis=1)
songs["new_entry"] = songs["count_at_first_obs"] == 1
songs["still_charting"] = songs["last_week"] == latest_week
songs["complete_run"] = songs["new_entry"] & ~songs["still_charting"]

print("songs:", len(songs))
print(songs[["new_entry", "still_charting", "complete_run"]].sum())
songs.head()

songs: 3117
new_entry         2924
still_charting     100
complete_run      2825
dtype: int64


,song_id,title,artist,first_week,last_week,weeks_in_window,weeks_on_chart_total,count_at_first_obs,debut_rank,best_rank,peak_pos,new_entry,still_charting,complete_run
0,020299|thatmexicanot,02.02.99,That Mexican OT,2024-01-27,2024-02-10,3,3,1,88,74,74,True,False,True
1,1000times|lildurk,1000 Times,Lil Durk Featuring Lil Baby,2025-04-12,2025-04-12,1,1,1,83,83,83,True,False,True
2,1035|tiesto,10:35,Tiesto Featuring Tate McRae,2023-02-04,2023-04-01,9,9,1,72,69,69,True,False,True
3,12to12|sombr,12 To 12,sombr,2025-08-16,2026-03-07,28,28,1,95,41,41,True,False,True
4,15minutes|sabrinacarpenter,15 Minutes,Sabrina Carpenter,2025-03-01,2025-03-01,1,1,1,67,67,67,True,False,True


## Cleaning: Spotify US Top 200 (weekly)

Steps:
1. Read in every weekly file and add a column for that week's chart date. The file names have the date in them, like `regional-us-weekly-2026-05-07.csv`.
2. Make `streams` a number and remove duplicate rows for the same track in the same week.
3. Make a matching key (cleaned title plus first artist listed) so we can join Spotify tracks to Billboard songs. The two sites write titles and artists differently. For example, Spotify puts "(feat. X)" in the title, and Billboard writes "Featuring" in the artist name while Spotify uses commas.

If you have not downloaded the Spotify files yet, set `USE_SPOTIFY = False` and the rest of the notebook will still run with Billboard only.

**Phase 3 change:** the matching key now also
- straightens curly apostrophes and quotes in titles, so "what’s wrong with me" matches "what's wrong with me"
- strips accents from artist names, so "beyoncé" matches "beyonce"
- fixes Billboard's "Feauring" typo, so the artist separator can find it

The Spotify settings also changed from one start date and a week count to a list of date ranges (`SPOTIFY_RANGES`). Today that list holds one run of 23 weeks. To add the October 2024 to October 2025 weeks for Hypothesis 3, we add one line to it.

In [4]:
# Spotify settings and the functions that build the title/artist matching key
USE_SPOTIFY = True
# Phase 3: a list of (first chart date, number of weeks) runs instead of one start date and a count,
# so more weeks can be added with one line, e.g. ("2024-10-03", 53),
SPOTIFY_RANGES = [
    ("2026-05-07", 23),  # Phase 3: 23 weeks (May 7 to Oct 8, 2026) instead of 20
]

def clean_title(titles):
    original = titles.astype(str).str.lower()
    cleaned = original
    # cut off extras like " (feat. X)", " [Explicit]", " - Remastered 2011"
    for mark in [" (", " [", " - "]:
        cleaned = cleaned.str.split(mark, regex=False).str[0]
    # Phase 3: curly apostrophes and quotes -> straight, so "what’s" matches "what's"
    cleaned = (cleaned.str.replace("\u2019", "'", regex=False)
                      .str.replace("\u2018", "'", regex=False)
                      .str.replace("\u201c", '"', regex=False)
                      .str.replace("\u201d", '"', regex=False))
    # take out punctuation
    for ch in [".", ",", "!", "?", "'", '"', ":", ";", "-", "*", "$", "/"]:
        cleaned = cleaned.str.replace(ch, "", regex=False)
    cleaned = cleaned.str.strip()
    return cleaned.where(cleaned != "", original)

# Phase 3: new helper. "beyoncé" -> "beyonce" (split accented letters apart and drop the accent marks)
def strip_accents(s):
    return s.str.normalize("NFKD").str.encode("ascii", errors="ignore").str.decode("ascii")

def clean_artist(artists):
    cleaned = strip_accents(artists.astype(str).str.lower())  # Phase 3: strip accents
    # Phase 3: fix a known Billboard typo so the separator below can find it
    cleaned = cleaned.str.replace(" feauring ", " featuring ", regex=False)
    # keep only the first artist listed
    for sep in [" featuring ", " feat. ", " feat ", ",", " with ", " x ", " & ", " and "]:
        cleaned = cleaned.str.split(sep, regex=False).str[0]
    for ch in [".", "'", '"', "-", "*", "$"]:
        cleaned = cleaned.str.replace(ch, "", regex=False)
    return cleaned.str.strip()

def make_key(titles, artists):
    return clean_title(titles) + "|" + clean_artist(artists)

songs["key"] = make_key(songs["title"], songs["artist"])

In [5]:
# Load all weekly Spotify files, clean streams, remove duplicates, and build the matching key
if USE_SPOTIFY:
    # Phase 3: every chart date in every run listed in SPOTIFY_RANGES
    spotify_dates = sorted(set().union(*[pd.date_range(start, periods=n_weeks, freq="7D")
                                         for start, n_weeks in SPOTIFY_RANGES]))

    frames = []
    for chart_date in spotify_dates:
        date_text = chart_date.strftime("%Y-%m-%d")
        one_week = pd.read_csv(DATA_FOLDER + "spotify_weekly_us/regional-us-weekly-" + date_text + ".csv")
        one_week.columns = one_week.columns.str.strip().str.lower().str.replace(" ", "_", regex=False)
        one_week = one_week.rename(columns={"track_name": "title", "artist_names": "artist"})
        one_week["chart_date"] = chart_date
        frames.append(one_week)
    sp = pd.concat(frames, ignore_index=True)

    print("columns:", list(sp.columns))
    sp["streams"] = pd.to_numeric(sp["streams"].astype(str).str.replace(",", "", regex=False), errors="coerce")

    rows_before = len(sp)
    sp = sp.drop_duplicates(["chart_date", "title", "artist"]).reset_index(drop=True)
    print("duplicate rows removed:", rows_before - len(sp))
    print("chart dates:", sp["chart_date"].min().date(), "to", sp["chart_date"].max().date(),
          "|", sp["chart_date"].nunique(), "weeks")
    print("missing streams:", sp["streams"].isna().sum())

    sp["key"] = make_key(sp["title"], sp["artist"])

columns: ['rank', 'uri', 'artist', 'title', 'source', 'peak_rank', 'previous_rank', 'weeks_on_chart', 'streams', 'chart_date']
duplicate rows removed: 0
chart dates: 2026-05-07 to 2026-10-08 | 23 weeks
missing streams: 0


### Streaming features and joining the two sources

For each Spotify track we record the week it first showed up on the Top 200, its rank and streams that week, how many weeks it stayed on the Top 200, its best rank, and its total streams. If two versions of a track (like clean and explicit) get the same key in the same week, we add their streams together.

Spotify only publishes the top 200, so "first week" means the week it first made the Top 200, not its release week. A track that first shows up in our very first week may have been charting before that, so we mark those with `spotify_first_in_window`.

Then we join these columns onto the Billboard song table using the key and check how many Billboard songs that debuted during our Spotify weeks found a match. A "matched" song is a Billboard song we also found in the Spotify files.

**Phase 3 change:** with more than one run of Spotify weeks, a track's first week is only trustworthy if we also loaded the week before it. `spotify_first_in_window` now checks exactly that. With our current single run it gives the same result as in Phase 2. We also print the top-40 Billboard debuts that still have no Spotify match.

In [6]:
# Build streaming features for each track and join them onto the Billboard songs
if USE_SPOTIFY:
    weekly = (sp.groupby(["key", "chart_date"], as_index=False)
                .agg(rank=("rank", "min"), streams=("streams", "sum"))
                .sort_values(["key", "chart_date"]))

    first_week_rows = (weekly.drop_duplicates("key", keep="first")
                             [["key", "chart_date", "rank", "streams"]]
                             .rename(columns={"chart_date": "spotify_first_date",
                                              "rank": "spotify_debut_rank",
                                              "streams": "streams_first_week"}))

    totals = (weekly.groupby("key", as_index=False)
                    .agg(spotify_weeks_top200=("chart_date", "nunique"),
                         spotify_best_rank=("rank", "min"),
                         spotify_total_streams=("streams", "sum")))

    spotify_features = first_week_rows.merge(totals, on="key")
    first_spotify_week = sp["chart_date"].min()
    # Phase 3: the first week counts only if the week before it was loaded (works with several date ranges)
    loaded_weeks = set(sp["chart_date"].unique())
    spotify_features["spotify_first_in_window"] = (
        (spotify_features["spotify_first_date"] - pd.Timedelta(days=7)).isin(loaded_weeks))

    merged = songs.merge(spotify_features, on="key", how="left")

    in_window = merged[merged["new_entry"]
                       & (merged["first_week"] >= first_spotify_week - pd.Timedelta(days=7))
                       & (merged["first_week"] <= sp["chart_date"].max())]
    print("Billboard new entries during our Spotify weeks:", len(in_window))
    print("share with a Spotify match:", round(in_window["streams_first_week"].notna().mean(), 3))
    # Phase 3: list the top-40 debuts that still have no match
    print("top-40 debuts still unmatched:")
    print(in_window.loc[in_window["streams_first_week"].isna() & (in_window["debut_rank"] <= 40),
                        ["title", "artist", "first_week", "debut_rank"]].to_string())
else:
    merged = songs.copy()
    print("Not using Spotify yet, so this is Billboard only.")

Billboard new entries during our Spotify weeks: 244
share with a Spotify match: 0.582
top-40 debuts still unmatched:
               title      artist first_week  debut_rank
259   Bass Persuades       Miley 2026-09-19          35
2730     This & That  Stray Kids 2026-08-22          38


**Phase 3 change:** the key fixes recovered all three spelling-related misses from Phase 2: the Beyoncé accent, the curly apostrophe, and the "Feauring" typo. Two top-40 debuts are still unmatched:
- Billboard credits "Bass Persuades" to "Miley", while Spotify credits it to "Miley Cyrus".
- Stray Kids' "This & That" never appears in our Spotify files.

We leave both unmatched instead of hand-coding exceptions. The match rate (58%) is close to Phase 2's 59%, and most unmatched songs debuted low on the Hot 100.

### Early signals and follow-up flags

We add the early signals we use as predictors. An early signal is anything we know about a song within its first two weeks on the Hot 100, before anyone knows how long it will last. We also add a follow-up flag that decides which songs we analyze and a flag for our outcome. All of them come from the weekly Billboard table.
- `week2_rank` and `week2_change`: the song's rank on the chart one week after its debut, and how many spots it moved (positive means it climbed). Both are missing when the song is not on that chart, or when we did not see its debut.
- `album_track`: True when the lead artist debuted 3 or more songs that week, which usually means an album came out.
- `prior_hits`: how many earlier songs by the same lead artist entered the Hot 100 in our window (since 2022). Phase 3 note: our tests now use the `established` flag below instead, because `prior_hits` cannot see hits from before 2022.
- `followed_1yr`: True for new entries that debuted at least 52 weeks before the latest chart, so we can see almost all of their run.
- `long_run`: True when a song spent 20 or more weeks on the chart.

**Phase 3 change:** the save step moved from this cell to the end of the notebook, so the files include the new Phase 3 columns.

In [7]:
# Add the early signals and follow-up flags (Phase 3: saving moved to the end of the notebook)
import numpy as np

# Week-2 rank and movement (positive week2_change = climbed)
# Phase 3: week 2 is the chart dated exactly 7 days after the debut. Before, we used the song's 2nd week on
# the chart, which for some songs was a re-entry weeks or months later.
week2 = bb[["song_id", "week", "rank"]].rename(columns={"week": "first_week", "rank": "week2_rank"})
week2["first_week"] = week2["first_week"] - pd.Timedelta(days=7)  # line each week up with the week before it
merged = merged.merge(week2, on=["song_id", "first_week"], how="left")
merged.loc[~merged["new_entry"], "week2_rank"] = np.nan
merged["week2_change"] = merged["debut_rank"] - merged["week2_rank"]

# Album track: the lead artist debuted 3+ songs in the same week
merged["lead_artist"] = merged["key"].str.split("|").str[1]
merged["same_week_debuts"] = merged.groupby(["lead_artist", "first_week"])["title"].transform("size")
merged["album_track"] = merged["new_entry"] & (merged["same_week_debuts"] >= 3)

# Artist history: earlier Hot 100 songs by the same lead artist since 2022
merged["prior_hits"] = (merged.groupby("lead_artist")["first_week"].rank(method="min") - 1).astype(int)

# Follow-up and outcome flags
merged["followed_1yr"] = merged["new_entry"] & (merged["first_week"] <= latest_week - pd.Timedelta(weeks=52))
merged["long_run"] = merged["weeks_on_chart_total"] >= 20

# New in Phase 3

These steps build the columns our preregistered hypotheses need. None of them existed in Phase 2.

## Week-2 Spotify streams and the front-loading ratio (Hypothesis 3)
For each Spotify track, starting from its first week in the Top 200:
- **`streams_week2`**: streams in the following week. It is 0 if the track fell out of the Top 200 (as preregistered), and missing if we did not load the following week.
- **`front_load` (F)**: first-week streams ÷ streams in the first two weeks. F = 1 means all early streams came in week 1 (a spike), F near 0.5 means steady streams, and F below 0.5 means growing streams.
- **`front_load_upper`**: our preregistered robustness version. When a track fell out in week 2, we replace its 0 with that week's No. 200 stream count, the most it could have had.
- **`streams_2wk`**: total streams in the first two weeks, used as a control, so we compare songs with the same total early streams.

**Phase 3 change (TA feedback):** a week outside the Top 200 is not a real stream count, so the analysis notebook's main test keeps only songs in the Top 200 both weeks (`fell_out_week2` = False). We keep the 0-stream and No. 200 versions above for the preregistered robustness check. This cell also records `spotify_lag_days`, the days between a song's first Spotify week and its Billboard debut, which the analysis notebook uses to keep only 9-day matches.

In [8]:
# Phase 3: week-2 streams, front-loading ratio F, and its robustness version
if USE_SPOTIFY:
    # Streams in the week right after each track's first Top 200 week
    sp_week2 = weekly.merge(first_week_rows[["key", "spotify_first_date"]], on="key")
    sp_week2 = sp_week2[sp_week2["chart_date"] == sp_week2["spotify_first_date"] + pd.Timedelta(days=7)]
    sp_week2 = sp_week2[["key", "streams"]].rename(columns={"streams": "streams_week2"})

    h3 = first_week_rows[["key", "spotify_first_date", "streams_first_week"]].merge(sp_week2, on="key", how="left")
    next_week = h3["spotify_first_date"] + pd.Timedelta(days=7)
    h3["week2_observed"] = next_week.isin(loaded_weeks)          # we loaded the following week
    h3["fell_out_week2"] = h3["week2_observed"] & h3["streams_week2"].isna()
    h3.loc[h3["fell_out_week2"], "streams_week2"] = 0           # preregistered: outside the Top 200 = 0 streams

    # Robustness: a missing week 2 gets that week's No. 200 stream count instead of 0
    floor = sp.groupby("chart_date")["streams"].min()
    h3["streams_week2_upper"] = np.where(h3["fell_out_week2"], next_week.map(floor), h3["streams_week2"])

    h3["streams_2wk"] = h3["streams_first_week"] + h3["streams_week2"]
    h3["front_load"] = h3["streams_first_week"] / h3["streams_2wk"]
    h3["front_load_upper"] = h3["streams_first_week"] / (h3["streams_first_week"] + h3["streams_week2_upper"])

    merged = merged.merge(h3.drop(columns=["spotify_first_date", "streams_first_week"]), on="key", how="left")
    # Phase 3: days from a song's first Spotify week to its Billboard debut (9 days is the normal gap, see Data description).
    # The analysis notebook keeps only 9-day matches, so this check now lives here with the rest of the cleaning.
    merged["spotify_lag_days"] = (merged["first_week"] - merged["spotify_first_date"]).dt.days
    print("smallest weekly No. 200 stream count:", int(floor.min()))
    print("tracks with an observed week 2:", int(h3["week2_observed"].sum()),
          "| fell out of the Top 200 in week 2:", int(h3["fell_out_week2"].sum()))

smallest weekly No. 200 stream count: 2018733
tracks with an observed week 2: 464 | fell out of the Top 200 in week 2: 128


## Established-artist flag (Hypothesis 2)
`established` = 1 if the song's lead artist appears anywhere on the Hot 100 in the **156 weeks (3 years) before the song's debut**, and 0 otherwise. It is only defined for new entries.

This is the one place we use pre-2022 charts, as our preregistration says. A debut in January 2022 needs to look back to January 2019, so we read the full raw file (`bb_raw`) here. The older archive has some reliability concerns about exact ranks, but we only ask whether the artist charted at all, so small rank errors do not change the flag.

This fixes the blind spot we noted for `prior_hits`, which only counts back to 2022, so a veteran whose hits all came earlier looked like a newcomer. One limit remains: the flag uses the lead artist only, so an artist whose earlier entries were all features counts as new.

**Phase 3 change (TA feedback):** our TA felt this flag is too generous, since one week anywhere on the Hot 100 in three years makes an artist established, and 84% of debuts qualify. We keep it as our preregistered flag and add two stricter versions for a robustness check: `established_3songs` (3 or more different Hot 100 songs as lead artist in the past 3 years, 66% of debuts) and `established_top10` (at least one top-10 hit as lead artist in the past 3 years, 48%).

In [9]:
# Phase 3: did the lead artist chart in the 3 years before this song's debut? (uses the full archive)
LOOKBACK_WEEKS = 156  # 3 years

bb_all = bb_raw.rename(columns={"chart_week": "week", "performer": "artist"})
bb_all["week"] = pd.to_datetime(bb_all["week"])
bb_all["lead_artist"] = clean_artist(bb_all["artist"])

# For each lead artist, a sorted array of every week they were on the chart
artist_weeks = {a: np.sort(grp["week"].unique()) for a, grp in bb_all.groupby("lead_artist")}

def charted_in_lookback(artist, debut):
    weeks = artist_weeks.get(artist)
    if weeks is None:
        return False
    start = np.datetime64(debut - pd.Timedelta(weeks=LOOKBACK_WEEKS))
    i = np.searchsorted(weeks, start, side="left")   # first chart week on or after the start of the look-back
    return i < len(weeks) and weeks[i] < np.datetime64(debut)

merged["established"] = [
    float(charted_in_lookback(a, d)) if new else np.nan
    for a, d, new in zip(merged["lead_artist"], merged["first_week"], merged["new_entry"])
]
print("established share among new entries:", round(merged.loc[merged["new_entry"], "established"].mean(), 3))

# Phase 3 (TA feedback: "too generous"): two stricter versions of `established` for a robustness check.
# For each lead artist: every week they charted as lead artist in the full archive, with the song and its rank
lead_rows = {a: grp[["week", "title", "current_week"]] for a, grp in bb_all.groupby("lead_artist")}

def lookback_record(artist, debut):
    """Number of different songs and best rank for this lead artist in the 3 years before `debut`."""
    rows = lead_rows.get(artist)
    if rows is None:
        return 0, np.nan
    start = debut - pd.Timedelta(weeks=LOOKBACK_WEEKS)
    window = rows[(rows["week"] >= start) & (rows["week"] < debut)]
    return window["title"].nunique(), window["current_week"].min()

record = [lookback_record(a, d) for a, d in zip(merged["lead_artist"], merged["first_week"])]
merged["lookback_songs"] = [r[0] for r in record]
merged["lookback_best_rank"] = [r[1] for r in record]
# Stricter flag 1: 3 or more different Hot 100 songs as lead artist in the past 3 years
merged["established_3songs"] = np.where(merged["new_entry"], (merged["lookback_songs"] >= 3).astype(float), np.nan)
# Stricter flag 2: at least one top-10 hit as lead artist in the past 3 years
merged["established_top10"] = np.where(merged["new_entry"], (merged["lookback_best_rank"] <= 10).astype(float), np.nan)
print(merged.loc[merged["new_entry"], ["established", "established_3songs", "established_top10"]].mean().round(3))

established share among new entries: 0.837


established           0.837
established_3songs    0.662
established_top10     0.477
dtype: float64


## Outcome for the linear regressions
`log_weeks` is the natural log of `weeks_on_chart_total`. Run lengths are very lopsided: most songs leave within a few weeks, and a few stay for 100 weeks or more (skew 2.7 in our Phase 2 EDA, where 0 would mean balanced). On a log scale, going from 1 to 2 weeks counts the same as going from 10 to 20, so a few very long runs cannot dominate the model. Our preregistration uses log(weeks) as the outcome for every linear regression.

In [10]:
# Phase 3: outcome for the linear regressions
merged["log_weeks"] = np.log(merged["weeks_on_chart_total"])

## Save the cleaned files
This is the Phase 2 save step, moved here so the files include the new columns. Phase 3 also saves the cleaned weekly Billboard and Spotify tables, which the analysis notebook uses for week-by-week plots.

After running this notebook, upload everything in `cleaned/` to `Data/cleaned/` in our GitHub repo.

In [11]:
# Save the cleaned files (moved from the Phase 2 cell above)
songs.to_csv(OUTPUT_FOLDER + "billboard_songs.csv", index=False)
merged.to_csv(OUTPUT_FOLDER + "analysis_ready.csv", index=False)
print("saved analysis_ready.csv with shape", merged.shape)

# Phase 3: also save the cleaned weekly tables for the analysis notebook
bb.to_csv(OUTPUT_FOLDER + "billboard_weekly_2022on.csv", index=False)
print("saved billboard_weekly_2022on.csv with shape", bb.shape)
if USE_SPOTIFY:
    sp.to_csv(OUTPUT_FOLDER + "spotify_weekly_us_clean.csv", index=False)
    print("saved spotify_weekly_us_clean.csv with shape", sp.shape)

saved analysis_ready.csv with shape (3117, 44)
saved billboard_weekly_2022on.csv with shape (25100, 8)
saved spotify_weekly_us_clean.csv with shape (4599, 11)


**A few rows of the final dataset (TA feedback).** Five random debuts from `analysis_ready.csv`:

In [12]:
# Phase 3 (TA feedback): a few rows of the final dataset (5 random new entries)
show = ["title", "artist", "first_week", "debut_rank", "week2_change", "album_track", "established",
        "weeks_on_chart_total", "long_run"]
merged.loc[merged["new_entry"], show].sample(5, random_state=1)

,title,artist,first_week,debut_rank,week2_change,album_track,established,weeks_on_chart_total,long_run
264,BBW,Drake,2026-05-30,74,NaN,True,1.0,1,False
748,Dos Dias,Tito Double P & Peso Pluma,2024-09-21,84,6.0,False,0.0,20,True
662,Dejame Entrar,Rauw Alejandro,2024-11-30,96,NaN,True,1.0,1,False
731,Don't Say You Love Me,Jin,2025-05-31,90,NaN,False,1.0,1,False
115,All Of You,"Stephanie Beatriz, Olga Merediz, John Leguizam...",2022-01-29,94,12.0,False,1.0,7,False
